# 📊 Análisis Exploratorio de Datos - Calidad de Datos
## Sistema de Monitoreo de Temperatura del aire - Antioquia

---

### Objetivo
Identificar problemas de calidad de datos en las observaciones de temperatura del aire registradas por estaciones en municipios de Antioquia, Colombia.

### Aspectos Analizados
- **Completitud de datos**: Datos ausentes, gaps temporales
- **Consistencia temporal**: Frecuencia de medición, regularidad
- **Outliers**: Valores extremos y anómalos (límites físicos, IQR, Z-score)
- **Duplicados**: Registros exactos y cuasi-duplicados
- **Calidad por dimensiones**: Estación, sensor, ubicación geográfica
- **Tendencias temporales**: Comparativo anual y patrones estacionales

### Tecnologías
- **Base de datos**: PostgreSQL 18.x
- **Análisis**: Python, Pandas, NumPy
- **Visualización**: Matplotlib, Seaborn, Plotly

### Período de análisis
- **Inicio:** 1 de abril de 2024
- **Fin:** 30 de abril de 2026

### Control de versiones
- 2.0.0 - Ampliación del rango de análisis (2024-04-01 → 2026-04-30). Migración a PostgreSQL 18.x. Optimización de queries mediante vistas materializadas. Implementación de Z-score. Análisis de tendencias temporales
- 1.0.1 - Implementación de formatos en las tablas de resultados
- 1.0.0 - Versión inicial del notebook

In [1]:
# Configuración inicial del notebook
__version__ = "2.0.0"
__date__ = "2026-05-29"

print("="*80)
print("✓ Notebook inicializado correctamente")
print(f"  Versión: {__version__}")
print(f"  Fecha: {__date__}")
print("="*80)

✓ Notebook inicializado correctamente
  Versión: 2.0.0
  Fecha: 2026-05-29


---

## 1️⃣ Configuración y Conexión

### 📦 Instalación de Dependencias

Primero instalamos todas las librerías necesarias en el entorno virtual.

In [2]:
print("\nInstalando librerías requeridas \n")
print("Instalando psycopg2-binary...")
%pip install -q psycopg2-binary
print("Instalando sqlalchemy...")
%pip install -q sqlalchemy
print("Instalando pandas...")
%pip install -q pandas
print("Instalando numpy...")
%pip install -q numpy
print("Instalando matplotlib...")
%pip install -q matplotlib
print("Instalando seaborn...")
%pip install -q seaborn
print("Instalando plotly...")
%pip install -q plotly
print("Instalando scipy...")
%pip install -q scipy
print("Instalando pytz...")
%pip install -q pytz
print("Instalando ipywidgets...")
%pip install -q ipywidgets
print("Instalando ipython...")
%pip install -q IPython
print("Instalando python-dotenv...")
%pip install -q python-dotenv

print("\nLibrerias instaladas correctamente")


Instalando librerías requeridas 

Instalando psycopg2-binary...


Note: you may need to restart the kernel to use updated packages.
Instalando sqlalchemy...
Note: you may need to restart the kernel to use updated packages.
Instalando pandas...
Note: you may need to restart the kernel to use updated packages.
Instalando numpy...
Note: you may need to restart the kernel to use updated packages.
Instalando matplotlib...
Note: you may need to restart the kernel to use updated packages.
Instalando seaborn...
Note: you may need to restart the kernel to use updated packages.
Instalando plotly...
Note: you may need to restart the kernel to use updated packages.
Instalando scipy...
Note: you may need to restart the kernel to use updated packages.
Instalando pytz...
Note: you may need to restart the kernel to use updated packages.
Instalando ipywidgets...
Note: you may need to restart the kernel to use updated packages.
Instalando ipython...
Note: you may need to restart the kernel to use updated packages.
Instalando python-dotenv...
Note: you may need to rest

### 📚 Importación de Librerías

En esta sección importamos todas las librerías necesarias para:
- **Conexión a base de datos**: `psycopg2`, `sqlalchemy`
- **Manipulación de datos**: `pandas`, `numpy`
- **Visualización**: `matplotlib`, `seaborn`, `plotly`
- **Análisis estadístico**: `scipy`
- **Manejo de fechas**: `datetime`
- **Gestión de credenciales**: `python-dotenv`
- **Configuración**: `warnings`, `os`

In [3]:
# Librerías para conexión a base de datos
import psycopg2
from sqlalchemy import create_engine, text
import sqlalchemy as sa

# Librerías para manipulación de datos
import pandas as pd
import numpy as np

# Librerías para visualización
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Librerías para análisis estadístico
from scipy import stats
from scipy.stats import zscore

# Manejo de fechas y tiempo
from datetime import datetime, timedelta
import pytz

# Utilidades
import warnings
import os
from typing import Dict, List, Tuple, Optional

# Gestión segura de credenciales
from dotenv import load_dotenv

# Para convertir pandas dataframes a tablas HTML
from IPython.core.display import HTML

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✓ Librerías importadas exitosamente")
print(f"  - pandas version: {pd.__version__}")
print(f"  - numpy version: {np.__version__}")
print(f"  - sqlalchemy version: {sa.__version__}")

✓ Librerías importadas exitosamente
  - pandas version: 3.0.3
  - numpy version: 2.4.6
  - sqlalchemy version: 2.0.50


## 🔌 Configuración y Conexión a la Base de Datos

Configuramos los parámetros de conexión a la base de datos **PostgreSQL 18.x** y establecemos la conexión.

In [4]:
# Cargar variables de entorno desde archivo .env
load_dotenv()

# Configuración de conexión a la base de datos
DB_CONFIG = {
    'host':     os.getenv('DB_HOST'),
    'port':     int(os.getenv('DB_PORT')),
    'database': os.getenv('DB_NAME'),
    'user':     os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
}

# Parámetros del análisis
PARAMS = {
    # Rango de fechas
    'fecha_inicio':     '2024-01-01',
    'fecha_fin':        '2026-04-30',

    # Umbrales físicos absolutos (°C)
    'temp_min_absoluta': -5.0,
    'temp_max_absoluta':  45.0,

    # Umbrales de rango normal para Antioquia (°C)
    'temp_min_normal':   5.0,
    'temp_max_normal':  35.0,

    # Detección de outliers
    'iqr_multiplicador':  1.5,
    'zscore_threshold':   3.0,

    # Gaps temporales
    'gap_threshold_horas': 3.0,

    # Cuasi-duplicados
    'ventana_minutos':    10,
    'delta_temp':          0.01,
}

print("✓ Parámetros de conexión cargados desde variables de entorno")
print("✓ Parámetros de análisis configurados")
print()
print("📅  Período de análisis:")
print(f"    Inicio : {PARAMS['fecha_inicio']}")
print(f"    Fin    : {PARAMS['fecha_fin']}")
print()
print("🌡️  Umbrales de temperatura:")
print(f"    Rango absoluto : {PARAMS['temp_min_absoluta']}°C → {PARAMS['temp_max_absoluta']}°C")
print(f"    Rango normal   : {PARAMS['temp_min_normal']}°C → {PARAMS['temp_max_normal']}°C")
print()
print("📅  Gaps temporales y cuasi-duplicados:")
print(f"    Umbral en horas para gaps temporales       : {PARAMS['gap_threshold_horas']}")
print(f"    Ventana en minutos para cuasi-duplicados   : {PARAMS['ventana_minutos']}")
print(f"    Delta de temperatura para cuasi-duplicados : {PARAMS['delta_temp']}")
print()
print("⚙️  Parámetros de detección:")
print(f"    Multiplicador IQR  : {PARAMS['iqr_multiplicador']}")
print(f"    Umbral Z-score     : {PARAMS['zscore_threshold']}")
print(f"    Umbral gap         : {PARAMS['gap_threshold_horas']} horas")
print()

✓ Parámetros de conexión cargados desde variables de entorno
✓ Parámetros de análisis configurados

📅  Período de análisis:
    Inicio : 2024-01-01
    Fin    : 2026-04-30

🌡️  Umbrales de temperatura:
    Rango absoluto : -5.0°C → 45.0°C
    Rango normal   : 5.0°C → 35.0°C

📅  Gaps temporales y cuasi-duplicados:
    Umbral en horas para gaps temporales       : 3.0
    Ventana en minutos para cuasi-duplicados   : 10
    Delta de temperatura para cuasi-duplicados : 0.01

⚙️  Parámetros de detección:
    Multiplicador IQR  : 1.5
    Umbral Z-score     : 3.0
    Umbral gap         : 3.0 horas



### 🛠️ Funciones Auxiliares

Definimos las funciones base que serán utilizadas a lo largo de todo el notebook:

- **`get_engine()`**: Crea el engine de conexión a la base de datos a partir de las variables de entorno configuradas en la celda anterior.
- **`ejecutar_query()`**: Ejecuta una query SQL y retorna un DataFrame de pandas, centralizando el manejo de errores y el log de resultados.
- **`build_where()`**: Construye cláusulas WHERE y AND con los filtros de fecha definidos en PARAMS, evitando la duplicación de este patrón a lo largo del notebook.

In [5]:
def get_engine():
    """Crea y retorna el engine de conexión a la base de datos."""
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    )
    return create_engine(url)


def ejecutar_query(query: str, descripcion: str = "") -> Optional[pd.DataFrame]:
    """
    Ejecuta una query SQL y retorna un DataFrame.

    Args:
        query       : SQL a ejecutar
        descripcion : Etiqueta para mensajes de log

    Returns:
        DataFrame con los resultados, o None si ocurre un error.
    """
    try:
        engine = get_engine()
        with engine.connect() as conn:
            df = pd.read_sql(text(query), conn)
        if descripcion:
            print(f"✓ Query ejecutada: {descripcion} ({len(df):,} filas)")
        return df
    except Exception as e:
        print(f"✗ Error ejecutando '{descripcion}': {e}")
        return None


def build_where(
    params: dict,
    alias: str = "",
    operator: str = "WHERE"
) -> str:
    """
    Construye una cláusula WHERE o AND con filtros de fecha.

    Args:
        params   : Diccionario PARAMS con 'fecha_inicio' y 'fecha_fin'
        alias    : Prefijo de tabla para la columna fecha.
                   Ej: "o" produce "o.fecha". Vacío produce "fecha".
        operator : "WHERE" para primera cláusula, "AND" para adicionales.

    Returns:
        String con la cláusula SQL, o cadena vacía si no hay filtros.

    Ejemplos:
        build_where(PARAMS)              → "WHERE fecha >= '2024-01-01' AND fecha <= '2026-04-30'"
        build_where(PARAMS, alias="o")   → "WHERE o.fecha >= '2024-01-01' AND o.fecha <= '2026-04-30'"
        build_where(PARAMS, operator="AND") → "AND fecha >= '2024-01-01' AND fecha <= '2026-04-30'"
    """
    col = f"{alias}.fecha" if alias else "fecha"
    clauses = []
    if params.get('fecha_inicio'):
        clauses.append(f"{col} >= '{params['fecha_inicio']}'")
    if params.get('fecha_fin'):
        clauses.append(f"{col} <= '{params['fecha_fin']}'")
    if not clauses:
        return ""
    return f"{operator} " + " AND ".join(clauses)


print("✓ Funciones auxiliares definidas")
print("   - get_engine()")
print("   - ejecutar_query(query, descripcion)")
print("   - build_where(params, alias, operator)")

✓ Funciones auxiliares definidas
   - get_engine()
   - ejecutar_query(query, descripcion)
   - build_where(params, alias, operator)


### 🔌 Verificación de Conexión

Verificamos que la conexión a la base de datos es exitosa y revisamos el estado del entorno:

- **Versión de PostgreSQL**: Confirma que estamos conectados a la instancia correcta.
- **Extensiones instaladas**: Lista las extensiones activas en la base de datos.
- **Vistas materializadas**: Muestra qué vistas del plan de optimización (Bloque B) están ya disponibles para ser consumidas por el notebook.

In [6]:
print("🔌 Verificando conexión a la base de datos...\n")

try:
    engine = get_engine()
    with engine.connect() as conn:

        # Versión de PostgreSQL
        result = conn.execute(text("SELECT version();"))
        version = result.fetchone()[0]
        print(f"✓ Conexión exitosa")
        print(f"  {version}\n")

        # Extensiones instaladas
        result = conn.execute(text("""
            SELECT extname, extversion
            FROM pg_extension
            ORDER BY extname;
        """))
        extensiones = result.fetchall()

        print("📦 Extensiones instaladas:")
        for ext in extensiones:
            print(f"   - {ext[0]} (v{ext[1]})")

        # Vistas materializadas disponibles
        result = conn.execute(text("""
            SELECT schemaname, matviewname
            FROM pg_matviews
            ORDER BY matviewname;
        """))
        vistas = result.fetchall()

        print(f"\n📋 Vistas materializadas disponibles:")
        if vistas:
            for v in vistas:
                print(f"   - {v[0]}.{v[1]}")
        else:
            print("   (ninguna aún)")

except Exception as e:
    print(f"✗ Error de conexión: {e}")

🔌 Verificando conexión a la base de datos...

✓ Conexión exitosa
  PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit

📦 Extensiones instaladas:
   - plpgsql (v1.0)

📋 Vistas materializadas disponibles:
   - public.mv_gaps_por_estacion
   - public.mv_intervalos
   - public.mv_inventario_geografico
   - public.mv_resumen_diario
   - public.mv_resumen_mensual
   - public.mv_stats_iqr


## 1️⃣ Inventario del Sistema de Monitoreo

En esta sección realizamos un inventario completo de la infraestructura de medición:

- **Distribución geográfica**: Departamentos, zonas hidrográficas, municipios y estaciones
- **Estaciones de medición**: Ubicación geográfica (latitud, longitud) y municipio asociado
- **Sensores disponibles**: Tipos de sensores registrados en el sistema

Toda la información de esta sección se obtiene de la vista materializada `mv_inventario_geografico` (**B-03**), que consolida el árbol geográfico `departamentos → zonas → municipios → estaciones` en una única estructura pre-calculada.

In [7]:
query_geo = """
SELECT
    departamento_nombre,
    zona_nombre,
    COUNT(DISTINCT municipio_id) AS total_municipios,
    COUNT(DISTINCT estacion_id)  AS total_estaciones
FROM mv_inventario_geografico
GROUP BY departamento_nombre, zona_nombre
ORDER BY departamento_nombre, zona_nombre;
"""

df_geo = ejecutar_query(query_geo, "distribución geográfica")

if df_geo is not None:
    print("📍 DISTRIBUCIÓN GEOGRÁFICA")
    display(HTML(df_geo.to_html()))
    print(f"\nRESUMEN:")
    print(f"  Total departamentos      : {df_geo['departamento_nombre'].nunique()}")
    print(f"  Total zonas hidrográficas: {df_geo['zona_nombre'].nunique()}")
    print(f"  Total municipios         : {df_geo['total_municipios'].sum()}")
    print(f"  Total estaciones         : {df_geo['total_estaciones'].sum()}")


✓ Query ejecutada: distribución geográfica (5 filas)
📍 DISTRIBUCIÓN GEOGRÁFICA


,departamento_nombre,zona_nombre,total_municipios,total_estaciones
0,ANTIOQUIA,ATRATO - DARIÉN,5,6
1,ANTIOQUIA,CARIBE - LITORAL,2,3
2,ANTIOQUIA,CAUCA,10,12
3,ANTIOQUIA,MEDIO MAGDALENA,8,9
4,ANTIOQUIA,NECHÍ,12,15



RESUMEN:
  Total departamentos      : 1
  Total zonas hidrográficas: 5
  Total municipios         : 37
  Total estaciones         : 45


### 🔧 Inventario de Estaciones

Detallamos las estaciones de medición activas en el sistema.


In [8]:
query_estaciones = """
SELECT
    estacion_id,
    estacion_nombre,
    municipio_nombre,
    zona_nombre,
    departamento_nombre,
    ROUND(estacion_latitud::numeric,  6) AS latitud,
    ROUND(estacion_longitud::numeric, 6) AS longitud
FROM mv_inventario_geografico
ORDER BY departamento_nombre, municipio_nombre, estacion_nombre;
"""

df_estaciones = ejecutar_query(query_estaciones, "estaciones")

if df_estaciones is not None:
    print("🔬 ESTACIONES DE MEDICIÓN")
    display(HTML(df_estaciones.to_html()))
    print(f"\nTotal estaciones: {len(df_estaciones)}")


✓ Query ejecutada: estaciones (45 filas)
🔬 ESTACIONES DE MEDICIÓN


,estacion_id,estacion_nombre,municipio_nombre,zona_nombre,departamento_nombre,latitud,longitud
0,1111500036,ABRIAQUI,ABRIAQUÍ,ATRATO - DARIÉN,ANTIOQUIA,6.64,-76.07
1,2701500213,ALTO DE LA CRUZ,AMALFI,NECHÍ,ANTIOQUIA,6.91,-75.08
2,0027010850,AMALFI,AMALFI,NECHÍ,ANTIOQUIA,6.91,-75.08
3,2702500107,ANGOSTURA,ANGOSTURA,NECHÍ,ANTIOQUIA,6.90,-75.33
4,0027025030,ANORI,ANORÍ,NECHÍ,ANTIOQUIA,7.07,-75.15
5,2620500209,ACUEDUCTO ARMENIA,ARMENIA,CAUCA,ANTIOQUIA,6.16,-75.78
6,0027015310,METROMEDELLIN,BELLO,NECHÍ,ANTIOQUIA,6.33,-75.55
7,0027015260,LA SALADA,CALDAS,NECHÍ,ANTIOQUIA,6.05,-75.62
8,0011115020,CAÑASGORDAS,CAÑASGORDAS,ATRATO - DARIÉN,ANTIOQUIA,6.76,-76.03
9,0012015060,TULENAPA,CAREPA,CARIBE - LITORAL,ANTIOQUIA,7.77,-76.67



Total estaciones: 45


### 📅 Período Temporal de los Datos

Analizamos el rango de fechas cubierto por las observaciones y el volumen de datos por período.


In [10]:
query_periodo = f"""
SELECT
    MIN(dia) primera_observacion,
    MAX(dia) ultima_observacion,
    MAX(dia) - MIN(dia) duracion_dias,
    SUM(num_observaciones)  total_observaciones,
    COUNT(DISTINCT estacion_id) estaciones_con_datos,
    COUNT(DISTINCT dia) dias_con_datos
FROM mv_resumen_diario
{build_where(PARAMS, alias='', operator='WHERE').replace('fecha', 'dia')};
"""

df_periodo = ejecutar_query(query_periodo, "período temporal")

if df_periodo is not None:
    total_obs = df_periodo['total_observaciones'].iloc[0]
    dias      = df_periodo['dias_con_datos'].iloc[0]

    print("📅 PERÍODO TEMPORAL DE LOS DATOS")
    print()
    print(f"  Primera observación         : {df_periodo['primera_observacion'].iloc[0]}")
    print(f"  Última observación          : {df_periodo['ultima_observacion'].iloc[0]}")
    print(f"  Duración total en días      : {df_periodo['duracion_dias'].iloc[0]} días")
    print(f"\n  Total observaciones         : {total_obs:>15,}")
    print(f"  Días con datos              : {dias:>15,}")
    print(f"  Estaciones con datos        : {df_periodo['estaciones_con_datos'].iloc[0]:>15,}")
    if dias > 0:
        print(f"\n  Promedio observaciones/día  : {total_obs/dias:>15,.1f}")


✓ Query ejecutada: período temporal (1 filas)
📅 PERÍODO TEMPORAL DE LOS DATOS

  Primera observación         : 2024-04-01
  Última observación          : 2026-04-30
  Duración total en días      : 759 días

  Total observaciones         :     1,238,343.0
  Días con datos              :             714
  Estaciones con datos        :              45

  Promedio observaciones/día  :         1,734.4


In [11]:
query_volumen = f"""
SELECT
    DATE_TRUNC('month', dia)                AS mes,
    SUM(num_observaciones)                  AS num_observaciones,
    COUNT(DISTINCT estacion_id)             AS estaciones_activas,
    COUNT(DISTINCT dia)                     AS dias_con_datos,
    MIN(temp_minima)                        AS temp_minima,
    MAX(temp_maxima)                        AS temp_maxima,
    ROUND(AVG(temp_promedio)::numeric, 2)   AS temp_promedio
FROM mv_resumen_diario
{build_where(PARAMS, alias='', operator='WHERE').replace('fecha', 'dia')}
GROUP BY DATE_TRUNC('month', dia)
ORDER BY mes;
"""

df_volumen = ejecutar_query(query_volumen, "volumen por período")

if df_volumen is not None:
    print("📊 VOLUMEN DE OBSERVACIONES POR MES")
    print()

    # Formatear para visualización
    df_volumen_display = df_volumen.copy()
    df_volumen_display['mes']               = pd.to_datetime(df_volumen_display['mes']).dt.strftime('%Y-%m')
    df_volumen_display['num_observaciones'] = df_volumen_display['num_observaciones'].apply(lambda x: f'{x:,}')
    df_volumen_display['temp_minima']       = df_volumen_display['temp_minima'].apply(lambda x: f'{x:.2f}')
    df_volumen_display['temp_maxima']       = df_volumen_display['temp_maxima'].apply(lambda x: f'{x:.2f}')
    df_volumen_display['temp_promedio']     = df_volumen_display['temp_promedio'].apply(lambda x: f'{x:.2f}')

    display(HTML(df_volumen_display.to_html()))

    # Estadísticas generales
    df_volumen['num_observaciones'] = df_volumen['num_observaciones'].astype(int)
    idx_max = df_volumen['num_observaciones'].idxmax()
    idx_min = df_volumen['num_observaciones'].idxmin()

    print(f"\nESTADÍSTICAS GENERALES:")
    print(f"  Total meses con datos         : {len(df_volumen)}")
    print(f"  Promedio observaciones/mes    : {df_volumen['num_observaciones'].mean():,.0f}")
    print(f"  Mes con más observaciones     : {pd.to_datetime(df_volumen.loc[idx_max, 'mes']).strftime('%Y-%m')} ({df_volumen.loc[idx_max, 'num_observaciones']:,})")
    print(f"  Mes con menos observaciones   : {pd.to_datetime(df_volumen.loc[idx_min, 'mes']).strftime('%Y-%m')} ({df_volumen.loc[idx_min, 'num_observaciones']:,})")


✓ Query ejecutada: volumen por período (25 filas)
📊 VOLUMEN DE OBSERVACIONES POR MES



,mes,num_observaciones,estaciones_activas,dias_con_datos,temp_minima,temp_maxima,temp_promedio
0,2024-04,"26,600.0",4,26,0.00,32.90,21.74
1,2024-05,"21,351.0",1,31,17.50,32.90,22.81
2,2024-06,"18,974.0",1,28,17.10,33.80,22.54
3,2024-07,"28,564.0",2,31,0.00,33.40,22.37
4,2024-08,"42,753.0",2,31,0.00,49.00,20.45
5,2024-09,"32,610.0",13,30,0.00,42.00,21.19
6,2024-10,"28,699.0",34,31,0.00,38.00,20.09
7,2024-11,"55,715.0",34,29,0.00,50.00,20.08
8,2024-12,"62,249.0",34,31,0.00,50.00,20.52
9,2025-01,"61,168.0",34,31,0.00,47.90,20.64



ESTADÍSTICAS GENERALES:
  Total meses con datos         : 25
  Promedio observaciones/mes    : 49,534
  Mes con más observaciones     : 2025-10 (67,395)
  Mes con menos observaciones   : 2025-02 (11,812)


## 2️⃣ Análisis de Completitud de Datos

### 🔍 Datos Ausentes y Gaps Temporales

En esta sección analizamos:
- **Cobertura temporal**: Porcentaje de datos presentes vs esperados por estación
- **Gaps temporales**: Períodos sin observaciones por estación
- **Patrones de ausencia**: Identificación de patrones en los datos faltantes


### 📊 Resumen de Datos por Estación

Analizamos el volumen y cobertura temporal de observaciones para cada estación,
identificando cuáles presentan mayor cantidad de datos faltantes o períodos
de inactividad.


In [12]:
query_estaciones_datos = f"""
SELECT
    geo.estacion_id,
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    SUM(d.num_observaciones) total_observaciones,
    MIN(d.dia) primera_obs,
    MAX(d.dia) ultima_obs,
    MAX(d.dia) - MIN(d.dia) dias_operacion,
    COUNT(DISTINCT d.dia) dias_con_datos,
    ROUND(
        COUNT(DISTINCT d.dia) * 100.0 /
        NULLIF((MAX(d.dia) - MIN(d.dia) + 1), 0)
    , 1)                                    AS pct_cobertura
FROM mv_resumen_diario d
JOIN mv_inventario_geografico geo ON d.estacion_id = geo.estacion_id
{build_where(PARAMS, alias='d', operator='WHERE').replace('d.fecha', 'd.dia')}
GROUP BY geo.estacion_id, geo.estacion_nombre, geo.municipio_nombre, geo.zona_nombre
HAVING SUM(d.num_observaciones) > 0
ORDER BY total_observaciones DESC;
"""

df_estaciones_datos = ejecutar_query(query_estaciones_datos, "datos por estación")

if df_estaciones_datos is not None:
    print("📊 RESUMEN DE DATOS POR ESTACIÓN")
    print()

    # Formatear para visualización
    df_display = df_estaciones_datos.copy()
    df_display['total_observaciones'] = df_display['total_observaciones'].apply(lambda x: f'{x:,}')
    df_display['pct_cobertura']       = df_display['pct_cobertura'].apply(lambda x: f'{x:.1f}%')

    display(HTML(df_display.to_html()))

    # Estadísticas generales
    print(f"\nESTADÍSTICAS GENERALES:")
    print(f"  Total estaciones con datos        : {len(df_estaciones_datos)}")
    print(f"  Promedio observaciones/estación   : {df_estaciones_datos['total_observaciones'].mean():,.0f}")
    print(f"  Promedio días con datos           : {df_estaciones_datos['dias_con_datos'].mean():.1f}")
    print(f"  Promedio cobertura                : {df_estaciones_datos['pct_cobertura'].mean():.1f}%")
    print(f"  Estación con más datos            : {df_estaciones_datos.iloc[0]['estacion_nombre']} ({df_estaciones_datos.iloc[0]['total_observaciones']:,} obs)")
    print(f"  Estación con menos datos          : {df_estaciones_datos.iloc[-1]['estacion_nombre']} ({df_estaciones_datos.iloc[-1]['total_observaciones']:,} obs)")


✓ Query ejecutada: datos por estación (45 filas)
📊 RESUMEN DE DATOS POR ESTACIÓN



,estacion_id,estacion_nombre,municipio_nombre,zona_nombre,total_observaciones,primera_obs,ultima_obs,dias_operacion,dias_con_datos,pct_cobertura
0,0027015330,AEROPUERTO OLAYA HERRERA,MEDELLÍN,NECHÍ,"452,941.0",2024-04-01,2026-04-30,759,693,91.2%
1,0023085270,AEROPUERTO J.M. CORDOVA,RIONEGRO,MEDIO MAGDALENA,"423,365.0",2024-04-01,2026-04-30,759,621,81.7%
2,0027015310,METROMEDELLIN,BELLO,NECHÍ,"12,019.0",2024-09-04,2026-04-30,603,506,83.8%
3,0023085260,LA SELVA,RIONEGRO,MEDIO MAGDALENA,"12,019.0",2024-09-04,2026-04-30,603,506,83.8%
4,1111500036,ABRIAQUI,ABRIAQUÍ,ATRATO - DARIÉN,"12,003.0",2024-09-04,2026-04-30,603,506,83.8%
5,0027015320,ARAGON,SANTA ROSA DE OSOS,NECHÍ,"11,959.0",2024-09-04,2026-04-30,603,506,83.8%
6,0026255030,SANTA ISABEL VALDIVIA,VALDIVIA,CAUCA,"11,949.0",2024-09-04,2026-04-30,603,505,83.6%
7,1111500203,CAMPO ALEGRE,DABEIBA,ATRATO - DARIÉN,"11,713.0",2024-10-29,2026-04-30,548,497,90.5%
8,0027011100,LAS BRISAS,SEGOVIA,NECHÍ,"11,676.0",2024-10-29,2026-04-30,548,497,90.5%
9,0023105070,MACEO,YOLOMBÓ,MEDIO MAGDALENA,"11,675.0",2024-09-04,2026-04-30,603,505,83.6%



ESTADÍSTICAS GENERALES:
  Total estaciones con datos        : 45
  Promedio observaciones/estación   : 27,519
  Promedio días con datos           : 381.6
  Promedio cobertura                : 83.4%
  Estación con más datos            : AEROPUERTO OLAYA HERRERA (452,941.0 obs)
  Estación con menos datos          : OTRAMINA (2.0 obs)


### ⏳ Análisis de Gaps Temporales

Identificamos períodos sin observaciones en cada estación para detectar
interrupciones en la recolección de datos.

Un gap se define como un intervalo entre dos observaciones consecutivas
de la misma estación que supera el umbral configurado en PARAMS
(`gap_threshold_horas`).

In [13]:
if PARAMS['gap_threshold_horas'] == 3.0:
    query_gaps = """
    SELECT
        geo.estacion_nombre,
        geo.municipio_nombre,
        geo.zona_nombre,
        g.num_gaps,
        ROUND(g.gap_minimo_horas::numeric, 1) gap_minimo_horas,
        ROUND(g.gap_maximo_horas::numeric, 1) gap_maximo_horas,
        ROUND(g.gap_promedio_horas::numeric, 1) gap_promedio_horas,
        ROUND(g.total_horas_perdidas::numeric, 1) total_horas_perdidas
    FROM mv_gaps_por_estacion g
    JOIN mv_inventario_geografico geo ON g.estacion_id = geo.estacion_id
    ORDER BY g.num_gaps DESC;
    """
else:
    query_gaps = f"""
    SELECT
        geo.estacion_nombre,
        geo.municipio_nombre,
        geo.zona_nombre,
        COUNT(*) num_gaps,
        ROUND(MIN(intervalo_horas)::numeric, 1) gap_minimo_horas,
        ROUND(MAX(intervalo_horas)::numeric, 1) gap_maximo_horas,
        ROUND(AVG(intervalo_horas)::numeric, 1) gap_promedio_horas,
        ROUND(SUM(intervalo_horas)::numeric, 1) total_horas_perdidas
    FROM mv_intervalos i
    JOIN mv_inventario_geografico geo ON i.estacion_id = geo.estacion_id
    WHERE i.intervalo_horas > {PARAMS['gap_threshold_horas']}
    GROUP BY geo.estacion_nombre, geo.municipio_nombre, geo.zona_nombre
    ORDER BY num_gaps DESC;
    """

df_gaps = ejecutar_query(query_gaps, "gaps temporales por estación")

if df_gaps is not None:
    print("⏳ GAPS TEMPORALES POR ESTACIÓN")
    print(f"   Umbral aplicado: > {PARAMS['gap_threshold_horas']} horas")
    print()

    if len(df_gaps) == 0:
        print("✓ No se encontraron gaps temporales con el umbral configurado")
    else:
        display(HTML(df_gaps.to_html()))

        print(f"\nESTADÍSTICAS GENERALES:")
        print(f"  Estaciones con gaps           : {len(df_gaps)}")
        print(f"  Total gaps detectados         : {df_gaps['num_gaps'].sum():,}")
        print(f"  Estación más afectada         : {df_gaps.iloc[0]['estacion_nombre']} ({df_gaps.iloc[0]['num_gaps']:,} gaps)")
        print(f"  Gap máximo registrado         : {df_gaps['gap_maximo_horas'].max():.1f} horas")
        print(f"  Total horas perdidas          : {df_gaps['total_horas_perdidas'].sum():,.1f} horas")



✓ Query ejecutada: gaps temporales por estación (43 filas)
⏳ GAPS TEMPORALES POR ESTACIÓN
   Umbral aplicado: > 3.0 horas



,estacion_nombre,municipio_nombre,zona_nombre,num_gaps,gap_minimo_horas,gap_maximo_horas,gap_promedio_horas,total_horas_perdidas
0,SANTA BARBARA,SANTA BÁRBARA,CAUCA,334,4.00,1100.00,18.10,6050.00
1,BOQUERON DE TOYO,GIRALDO,ATRATO - DARIÉN,280,4.00,505.00,14.50,4068.00
2,LA PALMA DE COCO,SEGOVIA,NECHÍ,79,4.00,757.00,69.70,5503.00
3,LA ESPERANZA RADIO,NECHÍ,NECHÍ,78,4.00,2158.00,74.00,5770.00
4,NORMAL SANTA TERESA,FRONTINO,ATRATO - DARIÉN,73,4.00,697.00,54.90,4004.00
5,SONSON,SONSÓN,CAUCA,64,4.00,1767.00,96.40,6168.00
6,AEROPUERTO OLAYA HERRERA,MEDELLÍN,NECHÍ,61,3.50,504.00,36.20,2205.50
7,ANORI,ANORÍ,NECHÍ,49,4.00,505.00,44.90,2200.00
8,MACEO,YOLOMBÓ,MEDIO MAGDALENA,45,4.00,1100.00,59.20,2664.00
9,GRANJA EXPERIMENTAL EL NUS,SAN ROQUE,MEDIO MAGDALENA,42,4.00,1097.00,69.30,2910.00



ESTADÍSTICAS GENERALES:
  Estaciones con gaps           : 43
  Total gaps detectados         : 1,682
  Estación más afectada         : SANTA BARBARA (334 gaps)
  Gap máximo registrado         : 11902.0 horas
  Total horas perdidas          : 106,649.8 horas


### ⏱️ Análisis de Intervalos entre Observaciones

Analizamos la regularidad en la frecuencia de medición de cada estación,
identificando irregularidades en los intervalos entre observaciones consecutivas.

Un sistema de monitoreo saludable debería mostrar intervalos consistentes
entre observaciones. Desviaciones significativas pueden indicar:

- **Intervalos muy cortos**: Posibles duplicados o errores de registro
- **Intervalos muy largos**: Gaps temporales o fallos en la transmisión
- **Alta variabilidad**: Inestabilidad en la frecuencia de medición


In [14]:
query_intervalos = f"""
SELECT
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    COUNT(*) num_intervalos,
    ROUND(MIN(i.intervalo_minutos)::numeric, 1) intervalo_minimo_min,
    ROUND(MAX(i.intervalo_minutos)::numeric, 1) intervalo_maximo_min,
    ROUND(AVG(i.intervalo_minutos)::numeric, 1) intervalo_promedio_min,
    ROUND(
        PERCENTILE_CONT(0.50) WITHIN GROUP
        (ORDER BY i.intervalo_minutos)::numeric, 1
    ) intervalo_mediana_min,
    ROUND(STDDEV(i.intervalo_minutos)::numeric, 1) intervalo_stddev_min
FROM mv_intervalos i
JOIN mv_inventario_geografico geo ON i.estacion_id = geo.estacion_id
WHERE i.intervalo_minutos IS NOT NULL
  AND i.intervalo_inicio >= '{PARAMS['fecha_inicio']}'
  AND i.intervalo_inicio <= '{PARAMS['fecha_fin']}'
GROUP BY geo.estacion_nombre, geo.municipio_nombre, geo.zona_nombre
ORDER BY intervalo_promedio_min;
"""

df_intervalos = ejecutar_query(query_intervalos, "intervalos entre observaciones")

if df_intervalos is not None:
    print("⏱️  INTERVALOS ENTRE OBSERVACIONES POR ESTACIÓN")
    print()

    display(HTML(df_intervalos.to_html()))

    print(f"\nESTADÍSTICAS GENERALES:")
    print(f"  Estaciones analizadas             : {len(df_intervalos)}")
    print(f"  Intervalo promedio global         : {df_intervalos['intervalo_promedio_min'].mean():.1f} min")
    print(f"  Intervalo mediana global          : {df_intervalos['intervalo_mediana_min'].median():.1f} min")
    print(f"  Estación más regular              : {df_intervalos.iloc[0]['estacion_nombre']} (stddev: {df_intervalos.iloc[0]['intervalo_stddev_min']:.1f} min)")
    print(f"  Estación menos regular            : {df_intervalos.iloc[-1]['estacion_nombre']} (stddev: {df_intervalos.iloc[-1]['intervalo_stddev_min']:.1f} min)")


✓ Query ejecutada: intervalos entre observaciones (45 filas)
⏱️  INTERVALOS ENTRE OBSERVACIONES POR ESTACIÓN



,estacion_nombre,municipio_nombre,zona_nombre,num_intervalos,intervalo_minimo_min,intervalo_maximo_min,intervalo_promedio_min,intervalo_mediana_min,intervalo_stddev_min
0,AEROPUERTO OLAYA HERRERA,MEDELLÍN,NECHÍ,452226,0.00,30242.00,2.40,2.00,57.70
1,AEROPUERTO J.M. CORDOVA,RIONEGRO,MEDIO MAGDALENA,422737,0.00,143568.00,2.60,2.00,227.60
2,OTRAMINA,TITIRIBÍ,CAUCA,1,60.00,60.00,60.00,60.00,NaN
3,TURBO,TURBO,CARIBE - LITORAL,2075,60.00,120.00,60.20,60.00,3.20
4,PARAMO BELMIRA,ENTRERRIOS,NECHÍ,1532,60.00,420.00,60.40,60.00,10.20
5,PISTA INDIRA,TURBO,CARIBE - LITORAL,2334,60.00,420.00,61.10,60.00,11.60
6,CAÑASGORDAS,CAÑASGORDAS,ATRATO - DARIÉN,3570,60.00,1500.00,62.80,60.00,29.30
7,ITUANGO,ITUANGO,CAUCA,4244,60.00,5040.00,63.40,60.00,107.50
8,EL JARDIN CUENCA ALTA RIO RISARALDA,JARDÍN,CAUCA,7266,51.00,18780.00,64.00,60.00,226.40
9,RETIRO,RETIRO,MEDIO MAGDALENA,7066,60.00,19980.00,65.80,60.00,260.10



ESTADÍSTICAS GENERALES:
  Estaciones analizadas             : 45
  Intervalo promedio global         : 77.5 min
  Intervalo mediana global          : 60.0 min
  Estación más regular              : AEROPUERTO OLAYA HERRERA (stddev: 57.7 min)
  Estación menos regular            : VEGACHI (stddev: 2185.2 min)


### 📊 Distribución de Intervalos entre Observaciones

Analizamos la distribución estadística de los intervalos entre observaciones
para identificar la frecuencia de medición predominante en el sistema y
detectar comportamientos atípicos.

Una distribución saludable debería mostrar alta concentración alrededor
del intervalo de medición esperado, con poca dispersión. Distribuciones
multimodales o con alta dispersión indican inconsistencias en la
frecuencia de medición.


In [15]:
query_intervalos_dist = f"""
SELECT
    ROUND(i.intervalo_minutos::numeric, 0)      AS intervalo_minutos,
    COUNT(*)                                     AS frecuencia
FROM mv_intervalos i
WHERE i.intervalo_minutos IS NOT NULL
  AND i.intervalo_minutos <= 180
  AND i.intervalo_inicio >= '{PARAMS['fecha_inicio']}'
  AND i.intervalo_inicio <= '{PARAMS['fecha_fin']}'
GROUP BY ROUND(i.intervalo_minutos::numeric, 0)
ORDER BY intervalo_minutos;
"""

df_intervalos_dist = ejecutar_query(query_intervalos_dist, "distribución de intervalos")

if df_intervalos_dist is not None:
    print("📊 DISTRIBUCIÓN DE INTERVALOS ENTRE OBSERVACIONES")
    print(f"   (intervalos <= 180 minutos)")
    print()

    # Estadísticas de la distribución
    total            = df_intervalos_dist['frecuencia'].sum()
    intervalo_modal  = df_intervalos_dist.loc[
        df_intervalos_dist['frecuencia'].idxmax(), 'intervalo_minutos'
    ]
    pct_modal        = df_intervalos_dist['frecuencia'].max() / total * 100

    # Mediana ponderada por frecuencia
    intervalos_expandidos = np.repeat(
        df_intervalos_dist['intervalo_minutos'].values,
        df_intervalos_dist['frecuencia'].astype(int).values
    )
    mediana = np.median(intervalos_expandidos)

    print(f"\n  Total intervalos analizados   : {total:,}")
    print(f"  Moda                          : {intervalo_modal:.0f} minutos ({pct_modal:.1f}% del total)")
    print(f"  Mediana                       : {mediana:.1f} minutos")

    # Distribución acumulada por rangos
    rangos = [
        ( 0,   5,  "0 - 5 min    (posibles duplicados)"),
        ( 5,  15,  "5 - 15 min   (alta frecuencia)"),
        (15,  60,  "15 - 60 min  (frecuencia normal)"),
        (60, 180,  "60 - 180 min (baja frecuencia)"),
    ]

    print(f"\n  DISTRIBUCIÓN POR RANGOS:")
    for r_min, r_max, etiqueta in rangos:
        mask  = (df_intervalos_dist['intervalo_minutos'] >= r_min) & \
                (df_intervalos_dist['intervalo_minutos'] <  r_max)
        count = df_intervalos_dist.loc[mask, 'frecuencia'].sum()
        pct   = count / total * 100 if total > 0 else 0
        print(f"    {etiqueta}: {count:>10,} ({pct:>5.1f}%)")

    # Visualización
    fig = px.bar(
        df_intervalos_dist,
        x='intervalo_minutos',
        y='frecuencia',
        title='Distribución de Intervalos entre Observaciones',
        labels={
            'intervalo_minutos': 'Intervalo (minutos)',
            'frecuencia':        'Frecuencia'
        }
    )

    # Línea vertical para la moda
    fig.add_vline(
        x=intervalo_modal,
        line_dash='dash',
        line_color='red',
        line_width=2,
        annotation_text=f'Moda: {intervalo_modal:.0f} min ({pct_modal:.1f}%)',
        annotation_position='top right',
        annotation_font_color='red'
    )

    fig.update_layout(
        bargap=0.1,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        )
    )

    fig.show()

✓ Query ejecutada: distribución de intervalos (53 filas)
📊 DISTRIBUCIÓN DE INTERVALOS ENTRE OBSERVACIONES
   (intervalos <= 180 minutos)


  Total intervalos analizados   : 1,234,688
  Moda                          : 2 minutos (68.8% del total)
  Mediana                       : 2.0 minutos

  DISTRIBUCIÓN POR RANGOS:
    0 - 5 min    (posibles duplicados):    864,142 ( 70.0%)
    5 - 15 min   (alta frecuencia):      9,991 (  0.8%)
    15 - 60 min  (frecuencia normal):        558 (  0.0%)
    60 - 180 min (baja frecuencia):    359,309 ( 29.1%)


## 3️⃣ Análisis de Consistencia Temporal

### 🔄 Duplicados y Cuasi-duplicados

En esta sección analizamos la consistencia de los registros en el tiempo,
identificando:

- **Duplicados exactos**: Registros con idéntica estación, fecha y valor
- **Cuasi-duplicados**: Registros con valores iguales en ventanas de tiempo
  cortas, que pueden indicar errores de transmisión o congelamiento del sensor


In [16]:
query_duplicados_exactos = f"""
SELECT
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    o.fecha,
    o.valor temperatura,
    COUNT(*) num_repeticiones
FROM observaciones o
JOIN mv_inventario_geografico geo ON o.estacion_id = geo.estacion_id
{build_where(PARAMS, alias='o', operator='WHERE').replace('o.fecha', 'o.fecha')}
GROUP BY
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    o.fecha,
    o.valor
HAVING COUNT(*) > 1
ORDER BY num_repeticiones DESC, o.fecha DESC;
"""

df_duplicados = ejecutar_query(query_duplicados_exactos, "duplicados exactos")

if df_duplicados is not None:
    print("🔁 DUPLICADOS EXACTOS")
    print()

    if len(df_duplicados) == 0:
        print("✓ No se encontraron duplicados exactos en el período analizado")
    else:
        print(f"  Total grupos duplicados       : {len(df_duplicados):,}")
        print(f"  Total registros duplicados    : {df_duplicados['num_repeticiones'].sum():,}")
        print(f"  Máximo repeticiones           : {df_duplicados['num_repeticiones'].max():,}")
        print()
        display(HTML(df_duplicados.to_html()))


✓ Query ejecutada: duplicados exactos (0 filas)
🔁 DUPLICADOS EXACTOS

✓ No se encontraron duplicados exactos en el período analizado


### 🔍 Cuasi-duplicados

Identificamos registros con valores iguales en ventanas de tiempo cortas
dentro de la misma estación y sensor, lo que puede indicar:

- **Congelamiento del sensor**: El sensor queda reportando el mismo valor
  durante un período prolongado
- **Errores de transmisión**: El mismo registro se retransmite varias veces
  con marcas de tiempo ligeramente diferentes

Un registro se considera cuasi-duplicado cuando su valor es idéntico al de
la observación siguiente, dentro de una ventana de
`ventana_minutos` minutos configurada en PARAMS.

In [17]:
query_cuasi_duplicados = f"""
WITH observaciones_con_siguiente AS (
    SELECT
        estacion_id,
        fecha,
        valor,
        LEAD(fecha) OVER (
            PARTITION BY estacion_id
            ORDER BY fecha
        ) AS fecha_siguiente,
        LEAD(valor) OVER (
            PARTITION BY estacion_id
            ORDER BY fecha
        ) AS valor_siguiente
    FROM observaciones
    {build_where(PARAMS, operator='WHERE')}
)
SELECT
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    COUNT(*)  num_cuasi_duplicados,
    MIN(oc.fecha) primera_ocurrencia,
    MAX(oc.fecha) ultima_ocurrencia,
    ROUND(AVG(
        EXTRACT(EPOCH FROM (oc.fecha_siguiente - oc.fecha)) / 60.0
    )::numeric, 1)    diferencia_promedio_min,
    ROUND(MIN(oc.valor)::numeric, 2) temp_minima,
    ROUND(MAX(oc.valor)::numeric, 2) temp_maxima
FROM observaciones_con_siguiente oc
JOIN mv_inventario_geografico geo ON oc.estacion_id = geo.estacion_id
WHERE oc.valor = oc.valor_siguiente
  AND EXTRACT(EPOCH FROM (oc.fecha_siguiente - oc.fecha)) / 60.0
      <= {PARAMS['ventana_minutos']}
GROUP BY geo.estacion_nombre, geo.municipio_nombre, geo.zona_nombre
ORDER BY num_cuasi_duplicados DESC;
"""

df_cuasi_duplicados = ejecutar_query(query_cuasi_duplicados, "cuasi-duplicados")

if df_cuasi_duplicados is not None:
    print("🔍 CUASI-DUPLICADOS")
    print(f"   Ventana de tiempo aplicada: {PARAMS['ventana_minutos']} minutos")
    print()

    if len(df_cuasi_duplicados) == 0:
        print("✓ No se encontraron cuasi-duplicados en el período analizado")
    else:
        print(f"  Total cuasi-duplicados detectados : {df_cuasi_duplicados['num_cuasi_duplicados'].sum():,}")
        print(f"  Estaciones afectadas              : {len(df_cuasi_duplicados):,}")
        print(f"  Estación más afectada             : {df_cuasi_duplicados.iloc[0]['estacion_nombre']} ({df_cuasi_duplicados.iloc[0]['num_cuasi_duplicados']:,})")
        print()
        display(HTML(df_cuasi_duplicados.to_html()))

    print()

✓ Query ejecutada: cuasi-duplicados (2 filas)
🔍 CUASI-DUPLICADOS
   Ventana de tiempo aplicada: 10 minutos

  Total cuasi-duplicados detectados : 322,593
  Estaciones afectadas              : 2
  Estación más afectada             : AEROPUERTO OLAYA HERRERA (172,482)



,estacion_nombre,municipio_nombre,zona_nombre,num_cuasi_duplicados,primera_ocurrencia,ultima_ocurrencia,diferencia_promedio_min,temp_minima,temp_maxima
0,AEROPUERTO OLAYA HERRERA,MEDELLÍN,NECHÍ,172482,2024-04-01,2026-04-29 23:58:00,2.00,0.00,33.70
1,AEROPUERTO J.M. CORDOVA,RIONEGRO,MEDIO MAGDALENA,150111,2024-04-01,2026-04-29 23:36:00,2.00,0.00,37.00


## 4️⃣ Análisis Estadístico de Temperaturas

### 📊 Estadísticas Descriptivas por Estación

En esta sección analizamos el comportamiento estadístico de las temperaturas
registradas por cada estación, identificando:

- **Estadísticas descriptivas**: Media, mediana, desviación estándar,
  percentiles 25 y 75
- **Rango de temperaturas**: Mínimos y máximos registrados por estación
- **Variabilidad**: Estaciones con mayor y menor dispersión térmica


In [18]:
query_temp_stats = f"""
SELECT
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    COUNT(DISTINCT d.dia) dias_analizados,
    SUM(d.num_observaciones) total_observaciones,
    ROUND(MIN(d.temp_minima)::numeric, 2) temp_minima,
    ROUND(MAX(d.temp_maxima)::numeric, 2) temp_maxima,
    ROUND(AVG(d.temp_promedio)::numeric, 2) temp_promedio,
    ROUND(AVG(d.temp_stddev)::numeric, 2)   temp_stddev,
    ROUND(AVG(d.percentil_25)::numeric, 2)  percentil_25,
    ROUND(AVG(d.mediana)::numeric, 2)       mediana,
    ROUND(AVG(d.percentil_75)::numeric, 2)  percentil_75,
    ROUND((MAX(d.temp_maxima) - MIN(d.temp_minima))::numeric, 2) rango_termico
FROM mv_resumen_diario d
JOIN mv_inventario_geografico geo ON d.estacion_id = geo.estacion_id
{build_where(PARAMS, alias='d', operator='WHERE').replace('d.fecha', 'd.dia')}
GROUP BY geo.estacion_nombre, geo.municipio_nombre, geo.zona_nombre
ORDER BY temp_promedio DESC;
"""

df_temp_stats = ejecutar_query(query_temp_stats, "estadísticas de temperatura")

if df_temp_stats is not None:
    print("🌡️  ESTADÍSTICAS DESCRIPTIVAS DE TEMPERATURA POR ESTACIÓN")
    print()

    display(HTML(df_temp_stats.to_html()))

    print(f"\nESTADÍSTICAS GENERALES:")
    print(f"  Estación más cálida           : {df_temp_stats.iloc[0]['estacion_nombre']} ({df_temp_stats.iloc[0]['temp_promedio']:.2f}°C)")
    print(f"  Estación más fría             : {df_temp_stats.iloc[-1]['estacion_nombre']} ({df_temp_stats.iloc[-1]['temp_promedio']:.2f}°C)")
    print(f"  Mayor rango térmico           : {df_temp_stats.loc[df_temp_stats['rango_termico'].idxmax(), 'estacion_nombre']} ({df_temp_stats['rango_termico'].max():.2f}°C)")
    print(f"  Mayor variabilidad (stddev)   : {df_temp_stats.loc[df_temp_stats['temp_stddev'].idxmax(), 'estacion_nombre']} ({df_temp_stats['temp_stddev'].max():.2f}°C)")
    print(f"  Menor variabilidad (stddev)   : {df_temp_stats.loc[df_temp_stats['temp_stddev'].idxmin(), 'estacion_nombre']} ({df_temp_stats['temp_stddev'].min():.2f}°C)")
    print()

✓ Query ejecutada: estadísticas de temperatura (45 filas)
🌡️  ESTADÍSTICAS DESCRIPTIVAS DE TEMPERATURA POR ESTACIÓN



,estacion_nombre,municipio_nombre,zona_nombre,dias_analizados,total_observaciones,temp_minima,temp_maxima,temp_promedio,temp_stddev,percentil_25,mediana,percentil_75,rango_termico
0,PISTA INDIRA,TURBO,CARIBE - LITORAL,100,2335.00,22.40,34.20,28.27,1.91,26.71,27.44,29.85,11.80
1,LA ESPERANZA RADIO,NECHÍ,NECHÍ,347,7395.00,2.40,45.10,27.57,2.60,25.52,26.61,29.63,42.70
2,SENA - EL BAGRE,EL BAGRE,NECHÍ,407,9340.00,21.20,37.10,27.40,2.81,25.03,26.57,29.71,15.90
3,TULENAPA,CAREPA,CARIBE - LITORAL,171,3878.00,22.10,35.70,26.95,2.50,24.92,26.03,29.05,13.60
4,LA PALMA DE COCO,SEGOVIA,NECHÍ,368,7591.00,21.00,36.80,26.94,3.02,24.58,25.69,29.46,15.80
5,TURBO,TURBO,CARIBE - LITORAL,88,2076.00,21.90,33.60,26.66,2.48,24.57,25.73,28.76,11.70
6,HACIENDA COTOVE,SANTA FE DE ANTIOQUIA,CAUCA,465,10999.00,16.90,38.10,25.82,4.06,22.32,24.63,29.09,21.20
7,HACIENDA TUNEZ,FREDONIA,CAUCA,368,8581.00,17.80,37.20,25.42,3.97,22.00,23.81,28.99,19.40
8,AEROPUERTO OTU,REMEDIOS,MEDIO MAGDALENA,497,11636.00,17.80,32.90,24.38,2.43,22.37,23.58,26.46,15.10
9,LAS BRISAS,SEGOVIA,NECHÍ,497,11676.00,19.00,33.10,24.32,2.25,22.45,23.67,26.10,14.10



ESTADÍSTICAS GENERALES:
  Estación más cálida           : PISTA INDIRA (28.27°C)
  Estación más fría             : PARAMO BELMIRA (10.55°C)
  Mayor rango térmico           : AEROPUERTO OLAYA HERRERA (50.00°C)
  Mayor variabilidad (stddev)   : HACIENDA COTOVE (4.06°C)
  Menor variabilidad (stddev)   : OTRAMINA (0.07°C)



## 5️⃣ Análisis de Outliers

### 🚨 Detección de Valores Atípicos

En esta sección identificamos observaciones con valores de temperatura
atípicos mediante tres métodos complementarios:

- **Límites físicos**: Valores fuera del rango absoluto posible para
  la región (`temp_min_absoluta`, `temp_max_absoluta`)
- **Rango normal**: Valores fuera del rango esperado para Antioquia
  (`temp_min_normal`, `temp_max_normal`)
- **IQR (Rango Intercuartílico)**: Valores que se alejan más de
  `iqr_multiplicador` veces el IQR por encima del percentil 75
  o por debajo del percentil 25, calculados por estación
- **Z-score**: Valores cuya distancia a la media supera
  `zscore_threshold` desviaciones estándar, calculados por estación


In [19]:
query_outliers_fisicos = f"""
SELECT
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    o.fecha,
    ROUND(o.valor::numeric, 2) temperatura,
    CASE
        WHEN o.valor < {PARAMS['temp_min_absoluta']}
            THEN 'Bajo límite físico'
        WHEN o.valor > {PARAMS['temp_max_absoluta']}
            THEN 'Sobre límite físico'
        WHEN o.valor < {PARAMS['temp_min_normal']}
            THEN 'Bajo rango normal'
        WHEN o.valor > {PARAMS['temp_max_normal']}
            THEN 'Sobre rango normal'
    END                                                 AS tipo_outlier
FROM observaciones o
JOIN mv_inventario_geografico geo ON o.estacion_id = geo.estacion_id
{build_where(PARAMS, alias='o', operator='WHERE').replace('o.fecha', 'o.fecha')}
  AND (
        o.valor < {PARAMS['temp_min_absoluta']}
     OR o.valor > {PARAMS['temp_max_absoluta']}
     OR o.valor < {PARAMS['temp_min_normal']}
     OR o.valor > {PARAMS['temp_max_normal']}
  )
ORDER BY geo.estacion_nombre, o.fecha;
"""

df_outliers_fisicos = ejecutar_query(query_outliers_fisicos, "outliers físicos y rango normal")

if df_outliers_fisicos is not None:
    print("🚨 OUTLIERS POR LÍMITES FÍSICOS Y RANGO NORMAL")
    print(f"   Límites físicos  : [{PARAMS['temp_min_absoluta']}°C, {PARAMS['temp_max_absoluta']}°C]")
    print(f"   Rango normal     : [{PARAMS['temp_min_normal']}°C, {PARAMS['temp_max_normal']}°C]")
    print()

    if len(df_outliers_fisicos) == 0:
        print("✓ No se encontraron outliers en el período analizado")
    else:
        # Resumen por tipo de outlier
        resumen_tipo = df_outliers_fisicos.groupby('tipo_outlier').agg(
            num_outliers  = ('temperatura', 'count'),
            temp_minima   = ('temperatura', 'min'),
            temp_maxima   = ('temperatura', 'max'),
            temp_promedio = ('temperatura', 'mean')
        ).round(2)

        print("  RESUMEN POR TIPO:")
        display(HTML(resumen_tipo.to_html()))

        # Resumen por estación
        resumen_estacion = df_outliers_fisicos.groupby(
            ['estacion_nombre', 'municipio_nombre', 'tipo_outlier']
        ).agg(
            num_outliers = ('temperatura', 'count')
        ).reset_index().sort_values('num_outliers', ascending=False)

        print("\n  RESUMEN POR ESTACIÓN:")
        display(HTML(resumen_estacion.to_html()))

        print(f"\nESTADÍSTICAS GENERALES:")
        print(f"  Total outliers detectados     : {len(df_outliers_fisicos):,}")
        print(f"  Estaciones afectadas          : {df_outliers_fisicos['estacion_nombre'].nunique():,}")



✓ Query ejecutada: outliers físicos y rango normal (3,849 filas)
🚨 OUTLIERS POR LÍMITES FÍSICOS Y RANGO NORMAL
   Límites físicos  : [-5.0°C, 45.0°C]
   Rango normal     : [5.0°C, 35.0°C]

  RESUMEN POR TIPO:


,num_outliers,temp_minima,temp_maxima,temp_promedio
tipo_outlier,,,,
Bajo rango normal,3405,0.00,4.90,1.56
Sobre límite físico,31,45.10,50.00,48.23
Sobre rango normal,413,35.10,45.00,36.60



  RESUMEN POR ESTACIÓN:


,estacion_nombre,municipio_nombre,tipo_outlier,num_outliers
1,AEROPUERTO J.M. CORDOVA,RIONEGRO,Bajo rango normal,1765
4,AEROPUERTO OLAYA HERRERA,MEDELLÍN,Bajo rango normal,1448
8,HACIENDA COTOVE,SANTA FE DE ANTIOQUIA,Sobre rango normal,214
7,ARAGON,SANTA ROSA DE OSOS,Bajo rango normal,178
17,SENA - EL BAGRE,EL BAGRE,Sobre rango normal,44
9,HACIENDA TUNEZ,FREDONIA,Sobre rango normal,40
13,LA PALMA DE COCO,SEGOVIA,Sobre rango normal,40
3,AEROPUERTO J.M. CORDOVA,RIONEGRO,Sobre rango normal,27
6,AEROPUERTO OLAYA HERRERA,MEDELLÍN,Sobre rango normal,26
2,AEROPUERTO J.M. CORDOVA,RIONEGRO,Sobre límite físico,18



ESTADÍSTICAS GENERALES:
  Total outliers detectados     : 3,849
  Estaciones afectadas          : 14


### 📊 Outliers por Rango Intercuartílico (IQR)

Identificamos valores atípicos usando el método IQR, que define como outlier
cualquier observación que se aleje más de `iqr_multiplicador` veces el IQR
por encima del percentil 75 o por debajo del percentil 25.

Los límites de detección se calculan por estación:

- **Límite inferior**: `percentil_25 - (iqr_multiplicador × IQR)`
- **Límite superior**: `percentil_75 + (iqr_multiplicador × IQR)`


In [20]:
query_outliers_iqr = f"""
SELECT
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    o.fecha,
    ROUND(o.valor::numeric, 2) temperatura,
    ROUND(st.percentil_25::numeric, 2) percentil_25,
    ROUND(st.percentil_75::numeric, 2) percentil_75,
    ROUND(st.iqr::numeric, 2)          iqr,
    ROUND((st.percentil_25 - ({PARAMS['iqr_multiplicador']} * st.iqr))::numeric, 2) limite_inferior,
    ROUND((st.percentil_75 + ({PARAMS['iqr_multiplicador']} * st.iqr))::numeric, 2) limite_superior,
    CASE
        WHEN o.valor < st.percentil_25 - ({PARAMS['iqr_multiplicador']} * st.iqr)
            THEN 'Bajo límite IQR'
        WHEN o.valor > st.percentil_75 + ({PARAMS['iqr_multiplicador']} * st.iqr)
            THEN 'Sobre límite IQR'
    END                                                         AS tipo_outlier
FROM observaciones o
JOIN mv_stats_iqr st ON o.estacion_id = st.estacion_id
JOIN mv_inventario_geografico geo ON o.estacion_id = geo.estacion_id
{build_where(PARAMS, alias='o', operator='WHERE').replace('o.fecha', 'o.fecha')}
  AND (
        o.valor < st.percentil_25 - ({PARAMS['iqr_multiplicador']} * st.iqr)
     OR o.valor > st.percentil_75 + ({PARAMS['iqr_multiplicador']} * st.iqr)
  )
ORDER BY geo.estacion_nombre, o.fecha;
"""

df_outliers_iqr = ejecutar_query(query_outliers_iqr, "outliers IQR")

if df_outliers_iqr is not None:
    print("📊 OUTLIERS POR RANGO INTERCUARTÍLICO (IQR)")
    print(f"   Multiplicador IQR: {PARAMS['iqr_multiplicador']}")
    print()

    if len(df_outliers_iqr) == 0:
        print("✓ No se encontraron outliers IQR en el período analizado")
    else:
        # Resumen por tipo de outlier
        resumen_tipo = df_outliers_iqr.groupby('tipo_outlier').agg(
            num_outliers  = ('temperatura', 'count'),
            temp_minima   = ('temperatura', 'min'),
            temp_maxima   = ('temperatura', 'max'),
            temp_promedio = ('temperatura', 'mean')
        ).round(2)

        print("  RESUMEN POR TIPO:")
        display(HTML(resumen_tipo.to_html()))

        # Resumen por estación
        resumen_estacion = df_outliers_iqr.groupby(
            ['estacion_nombre', 'municipio_nombre', 'tipo_outlier']
        ).agg(
            num_outliers  = ('temperatura', 'count'),
            limite_inferior = ('limite_inferior', 'first'),
            limite_superior = ('limite_superior', 'first')
        ).round(2).reset_index().sort_values('num_outliers', ascending=False)

        print("\n  RESUMEN POR ESTACIÓN:")
        display(HTML(resumen_estacion.to_html()))

        print(f"\nESTADÍSTICAS GENERALES:")
        print(f"  Total outliers detectados     : {len(df_outliers_iqr):,}")
        print(f"  Estaciones afectadas          : {df_outliers_iqr['estacion_nombre'].nunique():,}")


✓ Query ejecutada: outliers IQR (468,021 filas)
📊 OUTLIERS POR RANGO INTERCUARTÍLICO (IQR)
   Multiplicador IQR: 1.5

  RESUMEN POR TIPO:


,num_outliers,temp_minima,temp_maxima,temp_promedio
tipo_outlier,,,,
Bajo límite IQR,201835,0.00,25.00,15.13
Sobre límite IQR,266186,12.20,50.00,24.43



  RESUMEN POR ESTACIÓN:


,estacion_nombre,municipio_nombre,tipo_outlier,num_outliers,limite_inferior,limite_superior
5,AEROPUERTO J.M. CORDOVA,RIONEGRO,Sobre límite IQR,105805,14.72,19.46
4,AEROPUERTO J.M. CORDOVA,RIONEGRO,Bajo límite IQR,99986,14.72,19.46
7,AEROPUERTO OLAYA HERRERA,MEDELLÍN,Sobre límite IQR,80557,18.02,25.86
6,AEROPUERTO OLAYA HERRERA,MEDELLÍN,Bajo límite IQR,28335,18.02,25.86
48,LA SELVA,RIONEGRO,Bajo límite IQR,4340,16.32,20.64
32,GRANJA EXPERIMENTAL EL NUS,SAN ROQUE,Bajo límite IQR,4003,20.95,25.31
26,CORRIENTES,SAN VICENTE,Bajo límite IQR,3668,15.31,19.69
38,HACIENDA TUNEZ,FREDONIA,Bajo límite IQR,3604,23.05,27.85
49,LA SELVA,RIONEGRO,Sobre límite IQR,3391,16.32,20.64
46,LA SALADA,CALDAS,Bajo límite IQR,3375,15.65,19.96



ESTADÍSTICAS GENERALES:
  Total outliers detectados     : 468,021
  Estaciones afectadas          : 44


### 📉 Outliers por Z-score

Identificamos valores atípicos usando el método Z-score, que mide cuántas
desviaciones estándar se aleja cada observación de la media de su estación.

Un registro se considera outlier cuando su Z-score supera el umbral
configurado en PARAMS (`zscore_threshold`):

- **Z-score superior**: `(valor - media) / desviacion > zscore_threshold`
- **Z-score inferior**: `(valor - media) / desviacion < -zscore_threshold`

A diferencia del método IQR, el Z-score es más sensible a distribuciones
simétricas y permite detectar outliers moderados que el IQR podría pasar
por alto.

In [21]:
query_outliers_zscore = f"""
SELECT
    geo.estacion_nombre,
    geo.municipio_nombre,
    geo.zona_nombre,
    o.fecha,
    ROUND(o.valor::numeric, 2) temperatura,
    ROUND(st.media::numeric, 2) media_estacion,
    ROUND(st.desviacion::numeric, 2) desviacion_estacion,
    ROUND(((o.valor - st.media) / NULLIF(st.desviacion, 0))::numeric, 3) zscore,
    CASE
        WHEN (o.valor - st.media) / NULLIF(st.desviacion, 0) > {PARAMS['zscore_threshold']}
            THEN 'Z-score superior'
        WHEN (o.valor - st.media) / NULLIF(st.desviacion, 0) < -{PARAMS['zscore_threshold']}
            THEN 'Z-score inferior'
    END                                                         AS tipo_outlier
FROM observaciones o
JOIN mv_stats_iqr st  ON o.estacion_id = st.estacion_id
JOIN mv_inventario_geografico geo ON o.estacion_id = geo.estacion_id
{build_where(PARAMS, alias='o', operator='WHERE').replace('o.fecha', 'o.fecha')}
  AND ABS((o.valor - st.media) / NULLIF(st.desviacion, 0)) > {PARAMS['zscore_threshold']}
ORDER BY geo.estacion_nombre, o.fecha;
"""

df_outliers_zscore = ejecutar_query(query_outliers_zscore, "outliers Z-score")

if df_outliers_zscore is not None:
    print("📉 OUTLIERS POR Z-SCORE")
    print(f"   Umbral Z-score: ± {PARAMS['zscore_threshold']}")
    print()

    if len(df_outliers_zscore) == 0:
        print("✓ No se encontraron outliers Z-score en el período analizado")
    else:
        # Resumen por tipo de outlier
        resumen_tipo = df_outliers_zscore.groupby('tipo_outlier').agg(
            num_outliers  = ('temperatura', 'count'),
            temp_minima   = ('temperatura', 'min'),
            temp_maxima   = ('temperatura', 'max'),
            zscore_maximo = ('zscore', lambda x: x.abs().max())
        ).round(3)

        print("  RESUMEN POR TIPO:")
        display(HTML(resumen_tipo.to_html()))

        # Resumen por estación
        resumen_estacion = df_outliers_zscore.groupby(
            ['estacion_nombre', 'municipio_nombre', 'tipo_outlier']
        ).agg(
            num_outliers  = ('temperatura', 'count'),
            zscore_maximo = ('zscore', lambda x: x.abs().max()),
            media_estacion = ('media_estacion', 'first'),
            desviacion_estacion = ('desviacion_estacion', 'first')
        ).round(3).reset_index().sort_values('num_outliers', ascending=False)

        print("\n  RESUMEN POR ESTACIÓN:")
        display(HTML(resumen_estacion.to_html()))

        print(f"\nESTADÍSTICAS GENERALES:")
        print(f"  Total outliers detectados     : {len(df_outliers_zscore):,}")
        print(f"  Estaciones afectadas          : {df_outliers_zscore['estacion_nombre'].nunique():,}")
        print(f"  Z-score máximo registrado     : {df_outliers_zscore['zscore'].abs().max():.3f}")


✓ Query ejecutada: outliers Z-score (384,581 filas)
📉 OUTLIERS POR Z-SCORE
   Umbral Z-score: ± 3.0

  RESUMEN POR TIPO:


,num_outliers,temp_minima,temp_maxima,zscore_maximo
tipo_outlier,,,,
Z-score inferior,145029,0.00,24.60,23.48
Z-score superior,239552,12.20,50.00,37.49



  RESUMEN POR ESTACIÓN:


,estacion_nombre,municipio_nombre,tipo_outlier,num_outliers,zscore_maximo,media_estacion,desviacion_estacion
5,AEROPUERTO J.M. CORDOVA,RIONEGRO,Z-score superior,98294,37.49,17.07,0.88
4,AEROPUERTO J.M. CORDOVA,RIONEGRO,Z-score inferior,75511,19.44,17.07,0.88
7,AEROPUERTO OLAYA HERRERA,MEDELLÍN,Z-score superior,71720,19.74,21.95,1.42
6,AEROPUERTO OLAYA HERRERA,MEDELLÍN,Z-score inferior,15361,15.44,21.95,1.42
48,LA SELVA,RIONEGRO,Z-score inferior,4097,12.21,18.49,0.74
49,LA SELVA,RIONEGRO,Z-score superior,3344,12.64,18.49,0.74
26,CORRIENTES,SAN VICENTE,Z-score inferior,3274,11.92,17.49,0.79
38,HACIENDA TUNEZ,FREDONIA,Z-score inferior,3249,8.55,25.42,0.89
27,CORRIENTES,SAN VICENTE,Z-score superior,3218,10.69,17.49,0.79
33,GRANJA EXPERIMENTAL EL NUS,SAN ROQUE,Z-score superior,2939,10.38,23.16,0.95



ESTADÍSTICAS GENERALES:
  Total outliers detectados     : 384,581
  Estaciones afectadas          : 43
  Z-score máximo registrado     : 37.490


## 6️⃣ Score de Calidad por Estación

### 🏆 Evaluación Integral de Calidad

En esta sección consolidamos los resultados de todas las secciones anteriores
en un score de calidad por estación, que permite identificar de forma rápida
y comparable cuáles estaciones presentan mayores problemas de calidad de datos.

El score se construye penalizando cada estación según la proporción de
registros problemáticos detectados en cada dimensión:

- **Cobertura temporal**: Penalización por días sin datos respecto al
  período de operación esperado
- **Gaps temporales**: Penalización por horas perdidas respecto al
  total de horas del período
- **Duplicados exactos**: Penalización por registros duplicados respecto
  al total de observaciones
- **Cuasi-duplicados**: Penalización por registros cuasi-duplicados
  respecto al total de observaciones
- **Outliers físicos**: Penalización por valores fuera de límites físicos
  y rango normal respecto al total de observaciones
- **Outliers IQR**: Penalización por valores fuera de límites IQR
  respecto al total de observaciones
- **Outliers Z-score**: Penalización por valores fuera del umbral Z-score
  respecto al total de observaciones

El score final es un valor entre 0 y 100, donde 100 representa
calidad perfecta. Los DataFrames de las secciones anteriores se
reutilizan directamente, sin lanzar queries adicionales a la base de datos.

In [22]:
# Reutilizar df_estaciones cargado en la sección 1 (A-03)
df_estaciones_lista = df_estaciones[['estacion_id', 'estacion_nombre', 'municipio_nombre']].copy()

# Inicializar score en 100 para todas las estaciones
df_score = df_estaciones_lista.copy()
df_score['score']                   = 100.0
df_score['penalizacion_cobertura']  = 0.0
df_score['penalizacion_gaps']       = 0.0
df_score['penalizacion_duplicados'] = 0.0
df_score['penalizacion_cuasi']      = 0.0
df_score['penalizacion_outliers_fisicos'] = 0.0
df_score['penalizacion_outliers_iqr']     = 0.0
df_score['penalizacion_outliers_zscore']  = 0.0

for _, row in df_score.iterrows():
    estacion  = row['estacion_nombre']
    idx       = df_score[df_score['estacion_nombre'] == estacion].index[0]

    # Total observaciones de la estación
    total_obs = df_estaciones_datos.loc[
        df_estaciones_datos['estacion_nombre'] == estacion, 'total_observaciones'
    ].values
    total_obs = total_obs[0] if len(total_obs) > 0 else 0
    if total_obs == 0:
        df_score.at[idx, 'score'] = 0.0
        continue

    # 1. Penalización por cobertura temporal (peso: 20%)
    pct_cobertura = df_estaciones_datos.loc[
        df_estaciones_datos['estacion_nombre'] == estacion, 'pct_cobertura'
    ].values
    if len(pct_cobertura) > 0:
        pen_cobertura = round((100 - pct_cobertura[0]) * 0.20, 2)
        df_score.at[idx, 'penalizacion_cobertura'] = pen_cobertura
        df_score.at[idx, 'score'] -= pen_cobertura

    # 2. Penalización por gaps temporales (peso: 20%)
    horas_perdidas = df_gaps.loc[
        df_gaps['estacion_nombre'] == estacion, 'total_horas_perdidas'
    ].values
    if len(horas_perdidas) > 0:
        dias_op = df_estaciones_datos.loc[
            df_estaciones_datos['estacion_nombre'] == estacion, 'dias_operacion'
        ].values[0]
        horas_totales = dias_op * 24 if dias_op > 0 else 1
        pen_gaps = round(min((horas_perdidas[0] / horas_totales) * 100 * 0.20, 20), 2)
        df_score.at[idx, 'penalizacion_gaps'] = pen_gaps
        df_score.at[idx, 'score'] -= pen_gaps

    # 3. Penalización por duplicados exactos (peso: 15%)
    if df_duplicados is not None and len(df_duplicados) > 0:
        num_dup = df_duplicados.loc[
            df_duplicados['estacion_nombre'] == estacion, 'num_repeticiones'
        ].sum()
        pen_dup = round(min((num_dup / total_obs) * 100 * 0.15, 15), 2)
        df_score.at[idx, 'penalizacion_duplicados'] = pen_dup
        df_score.at[idx, 'score'] -= pen_dup

    # 4. Penalización por cuasi-duplicados (peso: 15%)
    if df_cuasi_duplicados is not None and len(df_cuasi_duplicados) > 0:
        num_cuasi = df_cuasi_duplicados.loc[
            df_cuasi_duplicados['estacion_nombre'] == estacion, 'num_cuasi_duplicados'
        ].sum()
        pen_cuasi = round(min((num_cuasi / total_obs) * 100 * 0.15, 15), 2)
        df_score.at[idx, 'penalizacion_cuasi'] = pen_cuasi
        df_score.at[idx, 'score'] -= pen_cuasi

    # 5. Penalización por outliers físicos y rango normal (peso: 15%)
    if df_outliers_fisicos is not None and len(df_outliers_fisicos) > 0:
        num_out_fis = df_outliers_fisicos.loc[
            df_outliers_fisicos['estacion_nombre'] == estacion
        ].shape[0]
        pen_out_fis = round(min((num_out_fis / total_obs) * 100 * 0.15, 15), 2)
        df_score.at[idx, 'penalizacion_outliers_fisicos'] = pen_out_fis
        df_score.at[idx, 'score'] -= pen_out_fis

    # 6. Penalización por outliers IQR (peso: 10%)
    if df_outliers_iqr is not None and len(df_outliers_iqr) > 0:
        num_out_iqr = df_outliers_iqr.loc[
            df_outliers_iqr['estacion_nombre'] == estacion
        ].shape[0]
        pen_out_iqr = round(min((num_out_iqr / total_obs) * 100 * 0.10, 10), 2)
        df_score.at[idx, 'penalizacion_outliers_iqr'] = pen_out_iqr
        df_score.at[idx, 'score'] -= pen_out_iqr

    # 7. Penalización por outliers Z-score (peso: 5%)
    if df_outliers_zscore is not None and len(df_outliers_zscore) > 0:
        num_out_zscore = df_outliers_zscore.loc[
            df_outliers_zscore['estacion_nombre'] == estacion
        ].shape[0]
        pen_out_zscore = round(min((num_out_zscore / total_obs) * 100 * 0.05, 5), 2)
        df_score.at[idx, 'penalizacion_outliers_zscore'] = pen_out_zscore
        df_score.at[idx, 'score'] -= pen_out_zscore

# Asegurar que el score no sea negativo
df_score['score'] = df_score['score'].clip(lower=0).round(2)

# Clasificar calidad
df_score['clasificacion'] = pd.cut(
    df_score['score'],
    bins=  [  0,  60,  75,  90, 100],
    labels=['🔴 Crítica', '🟠 Baja', '🟡 Media', '🟢 Alta'],
    include_lowest=True
)

# Ordenar por score ascendente para priorizar estaciones problemáticas
df_score = df_score.sort_values('score', ascending=True).reset_index(drop=True)

print("🏆 SCORE DE CALIDAD POR ESTACIÓN")
print()
display(HTML(df_score.to_html()))

print(f"\nRESUMEN POR CLASIFICACIÓN:")
resumen_clasificacion = df_score.groupby('clasificacion', observed=True).agg(
    num_estaciones = ('estacion_nombre', 'count'),
    score_promedio = ('score', 'mean'),
    score_minimo   = ('score', 'min'),
    score_maximo   = ('score', 'max')
).round(2)
display(HTML(resumen_clasificacion.to_html()))


🏆 SCORE DE CALIDAD POR ESTACIÓN



,estacion_id,estacion_nombre,municipio_nombre,score,penalizacion_cobertura,penalizacion_gaps,penalizacion_duplicados,penalizacion_cuasi,penalizacion_outliers_fisicos,penalizacion_outliers_iqr,penalizacion_outliers_zscore,clasificacion
0,0026185020,MESOPOTAMIA,LA UNIÓN,66.13,13.60,13.71,0.00,0.00,0.00,5.44,1.12,🟠 Baja
1,0023105030,VEGACHI,VEGACHÍ,71.09,11.88,14.09,0.00,0.00,0.00,2.82,0.12,🟠 Baja
2,0026180180,SONSON,SONSÓN,76.54,7.48,8.52,0.00,0.00,0.01,5.83,1.62,🟡 Media
3,0012015060,TULENAPA,CAREPA,78.07,8.16,8.72,0.00,0.00,0.01,3.67,1.37,🟡 Media
4,0027030140,LA PALMA DE COCO,SEGOVIA,78.57,6.60,8.37,0.00,0.00,0.08,5.66,0.72,🟡 Media
5,0025021480,LA ESPERANZA RADIO,NECHÍ,80.17,7.36,8.77,0.00,0.00,0.03,2.88,0.79,🟡 Media
6,0023085270,AEROPUERTO J.M. CORDOVA,RIONEGRO,80.32,3.66,3.73,0.00,5.32,0.06,4.86,2.05,🟡 Media
7,0011110010,NORMAL SANTA TERESA,FRONTINO,80.86,6.54,8.53,0.00,0.00,0.00,3.44,0.63,🟡 Media
8,0026225060,HACIENDA COTOVE,SANTA FE DE ANTIOQUIA,83.04,4.60,4.82,0.00,0.00,0.29,5.04,2.21,🟡 Media
9,2702500107,ANGOSTURA,ANGOSTURA,83.18,5.48,5.88,0.00,0.00,0.00,4.01,1.45,🟡 Media



RESUMEN POR CLASIFICACIÓN:


,num_estaciones,score_promedio,score_minimo,score_maximo
clasificacion,,,,
🟠 Baja,2,68.61,66.13,71.09
🟡 Media,31,85.51,76.54,90.00
🟢 Alta,12,92.39,90.15,99.94


## 7️⃣ Reporte Final de Calidad de Datos

### 📋 Consolidación de Resultados

En esta sección consolidamos todos los hallazgos del análisis en un reporte
ejecutivo que resume el estado de calidad de los datos del sistema de
monitoreo de temperatura de Antioquia.

El reporte presenta:

- **Resumen del período**: Cobertura temporal y volumen de observaciones
- **Resumen del inventario**: Estaciones y municipios analizados
- **Hallazgos por dimensión**: Resultados de cada método de diagnóstico
- **Estaciones críticas**: Listado priorizado de estaciones que requieren
  atención inmediata
- **Recomendaciones**: Acciones sugeridas según los hallazgos

El reporte se construye íntegramente desde los DataFrames ya cargados
en memoria durante el análisis, sin lanzar queries adicionales a la
base de datos.

In [23]:
from datetime import datetime

print()
print("📋 REPORTE FINAL DE CALIDAD DE DATOS")
print("   Sistema de Monitoreo de Temperatura — Antioquia")
print(f"   Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Versión notebook: {__version__}")
print()

# -----------------------------------------------------------------------------
# 1. Resumen del período
# -----------------------------------------------------------------------------
print("\n1. RESUMEN DEL PERÍODO")
print()
if df_periodo is not None:
    print(f"   Período analizado     : {PARAMS['fecha_inicio']} → {PARAMS['fecha_fin']}")
    print(f"   Primera observación   : {df_periodo['primera_observacion'].iloc[0]}")
    print(f"   Última observación    : {df_periodo['ultima_observacion'].iloc[0]}")
    print(f"   Duración              : {df_periodo['duracion_dias'].iloc[0]} días")
    print(f"   Total observaciones   : {df_periodo['total_observaciones'].iloc[0]:,}")
    print(f"   Días con datos        : {df_periodo['dias_con_datos'].iloc[0]:,}")

# -----------------------------------------------------------------------------
# 2. Resumen del inventario
# -----------------------------------------------------------------------------
print("\n2. RESUMEN DEL INVENTARIO")
print()
if df_geo is not None:
    print(f"   Departamentos         : {df_geo['departamento_nombre'].nunique()}")
    print(f"   Zonas hidrográficas   : {df_geo['zona_nombre'].nunique()}")
    print(f"   Municipios            : {df_geo['total_municipios'].sum()}")
    print(f"   Estaciones            : {df_geo['total_estaciones'].sum()}")

# -----------------------------------------------------------------------------
# 3. Hallazgos por dimensión
# -----------------------------------------------------------------------------
print("\n3. HALLAZGOS POR DIMENSIÓN")
print()

# Cobertura temporal
if df_estaciones_datos is not None:
    estaciones_baja_cobertura = df_estaciones_datos[
        df_estaciones_datos['pct_cobertura'] < 80
    ]
    print(f"   Cobertura temporal:")
    print(f"     Cobertura promedio          : {df_estaciones_datos['pct_cobertura'].mean():.1f}%")
    print(f"     Estaciones con < 80%        : {len(estaciones_baja_cobertura):,}")

# Gaps temporales
if df_gaps is not None:
    print(f"\n   Gaps temporales (> {PARAMS['gap_threshold_horas']}h):")
    print(f"     Estaciones con gaps         : {len(df_gaps):,}")
    print(f"     Total horas perdidas        : {df_gaps['total_horas_perdidas'].sum():,.1f} h")
    print(f"     Gap máximo registrado       : {df_gaps['gap_maximo_horas'].max():.1f} h")

# Duplicados exactos
if df_duplicados is not None:
    print(f"\n   Duplicados exactos:")
    print(f"     Total grupos duplicados     : {len(df_duplicados):,}")
    print(f"     Total registros duplicados  : {df_duplicados['num_repeticiones'].sum():,}")

# Cuasi-duplicados
if df_cuasi_duplicados is not None:
    print(f"\n   Cuasi-duplicados (ventana {PARAMS['ventana_minutos']} min):")
    print(f"     Estaciones afectadas        : {len(df_cuasi_duplicados):,}")
    print(f"     Total cuasi-duplicados      : {df_cuasi_duplicados['num_cuasi_duplicados'].sum():,}")

# Outliers físicos y rango normal
if df_outliers_fisicos is not None:
    print(f"\n   Outliers físicos y rango normal:")
    print(f"     Total outliers detectados   : {len(df_outliers_fisicos):,}")
    print(f"     Estaciones afectadas        : {df_outliers_fisicos['estacion_nombre'].nunique():,}")
    resumen_tipo_fis = df_outliers_fisicos.groupby('tipo_outlier')['temperatura'].count()
    for tipo, count in resumen_tipo_fis.items():
        print(f"       {tipo:<30}: {count:,}")

# Outliers IQR
if df_outliers_iqr is not None:
    print(f"\n   Outliers IQR (multiplicador: {PARAMS['iqr_multiplicador']}):")
    print(f"     Total outliers detectados   : {len(df_outliers_iqr):,}")
    print(f"     Estaciones afectadas        : {df_outliers_iqr['estacion_nombre'].nunique():,}")

# Outliers Z-score
if df_outliers_zscore is not None:
    print(f"\n   Outliers Z-score (umbral: ±{PARAMS['zscore_threshold']}):")
    print(f"     Total outliers detectados   : {len(df_outliers_zscore):,}")
    print(f"     Estaciones afectadas        : {df_outliers_zscore['estacion_nombre'].nunique():,}")
    print(f"     Z-score máximo registrado   : {df_outliers_zscore['zscore'].abs().max():.3f}")

# -----------------------------------------------------------------------------
# 4. Estaciones críticas
# -----------------------------------------------------------------------------
print("\n4. ESTACIONES CRÍTICAS")
print("-"*80)
if df_score is not None:
    estaciones_criticas = df_score[df_score['clasificacion'] == '🔴 Crítica']
    estaciones_bajas    = df_score[df_score['clasificacion'] == '🟠 Baja']

    print(f"   Clasificación general:")
    print(f"     🟢 Alta    : {len(df_score[df_score['clasificacion'] == '🟢 Alta']):,} estaciones")
    print(f"     🟡 Media   : {len(df_score[df_score['clasificacion'] == '🟡 Media']):,} estaciones")
    print(f"     🟠 Baja    : {len(estaciones_bajas):,} estaciones")
    print(f"     🔴 Crítica : {len(estaciones_criticas):,} estaciones")

    if len(estaciones_criticas) > 0:
        print(f"\n   Estaciones con calidad crítica:")
        for _, row in estaciones_criticas.iterrows():
            print(f"     • {row['estacion_nombre']} ({row['municipio_nombre']}) — Score: {row['score']:.2f}")

    if len(estaciones_bajas) > 0:
        print(f"\n   Estaciones con calidad baja:")
        for _, row in estaciones_bajas.iterrows():
            print(f"     • {row['estacion_nombre']} ({row['municipio_nombre']}) — Score: {row['score']:.2f}")

# -----------------------------------------------------------------------------
# 5. Recomendaciones
# -----------------------------------------------------------------------------
print("\n5. RECOMENDACIONES")
print("-"*80)

if df_score is not None:
    if len(estaciones_criticas) > 0:
        print("   🔴 ACCIÓN INMEDIATA:")
        print("     • Revisar conectividad y estado físico de estaciones críticas")
        print("     • Verificar pipeline de transmisión de datos")

    if df_gaps is not None and df_gaps['total_horas_perdidas'].sum() > 0:
        print("\n   🟠 GAPS TEMPORALES:")
        print("     • Investigar interrupciones en estaciones con mayor acumulado de horas perdidas")
        print("     • Evaluar redundancia en canales de transmisión")

    if df_cuasi_duplicados is not None and df_cuasi_duplicados['num_cuasi_duplicados'].sum() > 0:
        print("\n   🟡 CUASI-DUPLICADOS:")
        print("     • Revisar configuración de frecuencia de muestreo en estaciones afectadas")
        print("     • Verificar posible congelamiento de sensores")

    if df_outliers_fisicos is not None and len(df_outliers_fisicos) > 0:
        print("\n   🟡 OUTLIERS:")
        print("     • Calibrar sensores en estaciones con alta proporción de outliers físicos")
        print("     • Revisar umbrales de rango normal según altitud y zona climática")

print()


📋 REPORTE FINAL DE CALIDAD DE DATOS
   Sistema de Monitoreo de Temperatura — Antioquia
   Generado: 2026-05-29 19:52:33
   Versión notebook: 2.0.0


1. RESUMEN DEL PERÍODO

   Período analizado     : 2024-01-01 → 2026-04-30
   Primera observación   : 2024-04-01
   Última observación    : 2026-04-30
   Duración              : 759 días
   Total observaciones   : 1,238,343.0
   Días con datos        : 714

2. RESUMEN DEL INVENTARIO

   Departamentos         : 1
   Zonas hidrográficas   : 5
   Municipios            : 37
   Estaciones            : 45

3. HALLAZGOS POR DIMENSIÓN

   Cobertura temporal:
     Cobertura promedio          : 83.4%
     Estaciones con < 80%        : 10

   Gaps temporales (> 3.0h):
     Estaciones con gaps         : 43
     Total horas perdidas        : 106,649.8 h
     Gap máximo registrado       : 11902.0 h

   Duplicados exactos:
     Total grupos duplicados     : 0
     Total registros duplicados  : 0

   Cuasi-duplicados (ventana 10 min):
     Estaciones afe